### 模型二：

In [1]:
import pandas as pd
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, LpBinary,LpStatus
import collections
from itertools import product



file_path = "realdata_v5_2_new.xlsx"  
activities_df = pd.read_excel(file_path, sheet_name="ActivitiesInfo")
classroom_df = pd.read_excel(file_path, sheet_name="ClassroomsInfo")
student_courses_df = pd.read_excel(file_path, sheet_name="StudentsInfo")



In [91]:
df = pd.read_csv("processed_schedule.csv")
# df = df[(df['Week'] == 1) & (df['Time_slot'] == 2)]
df

,Week,Time_slot,subject_candidates
0,1,1,"{18, 5, 15}"
1,1,2,{11}
2,1,3,"{16, 52, 13}"
3,1,4,"{53, 13}"
4,1,5,{48}
...,...,...,...
87,11,9,{10}
88,12,1,{2}
89,12,2,{33}
90,12,6,{6}


In [98]:
courses = student_courses_df["Course_ID"]
courses = set([int(num) for row in courses for num in row.split(', ')])

classrooms = classroom_df["Classroom_ID"]
classrooms = set(int(row) for row in classrooms)

activity_types = activities_df["Activity_Type"]
activity_types = set(activity_types)

In [99]:
# Many parameters in Model 2 are derived by reducing the dimensions of parameters from Model 1. 
# For example, Model 1 has a parameter Ysai, while Model 2 has a parameter Ysi, which removes the dimension 'a'.
# Parameters
Pc = [row["Classroom_ID"] for _, row in classroom_df.iterrows() if row["Has_Computers"] == 1]
Ysi = {(row["Course_ID"], row["Activity_Type"]): 1 for _, row in activities_df.iterrows()}
Gs = {row["Course_ID"]: 1 for _, row in activities_df.iterrows() if row["Requires_Separation"] == 1}
Cs = {row["Course_ID"]: 1 for _, row in activities_df.iterrows() if row["Requires_Computers"] == 1}
Ts = {row["Course_ID"]: 1 for _, row in activities_df.iterrows() if row["Requires_Tables"] == 1}
Tc = {row["Classroom_ID"]: 1 for _, row in classroom_df.iterrows() if row["Has_Tables"] == 1}
Act = {(row["Classroom_ID"], int(w), int(t)): 1 for _, row in classroom_df.iterrows() for w in row["Available_Weeks"].split(', ') for t in row["Time_Slot"].split(', ')} #
Ic = {row["Classroom_ID"]: 1 for _, row in classroom_df.iterrows() if row["Is_Isolated"] == 1}
Ns = {row["Course_ID"]: row["Num_Students"] for _, row in activities_df.iterrows()}
Msg = collections.defaultdict(int)
for _, row in student_courses_df.iterrows():
    class_id = row["Class_ID"]
    course_ids = list(map(int, str(row["Course_ID"]).split(', ')))  
    for course_id in course_ids:
        Msg[(course_id, class_id)] += 1  

            
class_courses = collections.defaultdict(set)
for (course, class_id), _ in Msg.items():
    class_courses[course].add(class_id)

Capacity = {row["Classroom_ID"]: row["Capacity"] for _, row in classroom_df.iterrows()}
Occupancy_rate = {1: 1, 2: 0.9, 3: 0}
CAPci = {(c, i): int(Capacity[c] * Occupancy_rate[i]) for c, i in product(classrooms, activity_types)}

In [107]:
import csv
with open("results-虚拟教室.csv", "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Week", "Time_slot", "subject_candidates", "result","value"])

results = []

In [108]:


for count in range(len(df)):
    week = df.iloc[count]["Week"]
    time_slot = df.iloc[count]["Time_slot"]
    subjects = df["subject_candidates"].tolist()
    subjects = [int(i.strip('{}')) for i in subjects[count].split(', ')]
    # 2. define the model
    model = LpProblem(name="classroom_assignment", sense=LpMinimize)

    z = {(c, s): LpVariable(f"z_{c}_{s}", cat=LpBinary) for c in classrooms for s in subjects}
    w = {(c, s, g): LpVariable(f"w_{c}_{s}_{g}", cat=LpBinary) for c in classrooms for s in subjects for g in class_courses[s]}
   
    # add slack variable
    eta = {(s): LpVariable(f"eta_{s}", lowBound=0) for s in subjects}
    lamda = 50
    beta = 4
    # add virtual classroom
    z_v = {(s): LpVariable(f"z_v_{s}", cat=LpBinary) for s in subjects}

    # objective function
    model += (
    lpSum(z[c, s] for c in classrooms for s in subjects) + 
    beta * lpSum(z_v[s] for s in subjects),
    "Minimize_Classroom_Usage"
)
    # A.13 - make sure the classroom capacity is enough for each subject
  
    for s in subjects:
        model += lpSum(z[c, s] * sum(CAPci.get((c, i), 0) * Ysi.get((s, i), 0) for i in activity_types) for c in classrooms) >= Ns[s]* (1 - z_v[s]), f"Capacity_Constraint_{s}"
    
    # A.14 - make sure the classroom capacity is enough for each class
    # for s in subjects:
    #     if Ts.get(s, 0) == 1:
    #         for g in class_courses[s]:
    #             model += lpSum(w[c, s, g] * sum(CAPci.get((c, i), 0) * Ysi.get((s, i), 0) for i in activity_types) for c in classrooms) >= Msg.get((s, g), 0), f"Classroom_Capacity_{s}_{g}"
    for s in subjects:
        if Ts.get(s, 0) == 1 :
            for g in class_courses[s]:
                model +=  lpSum(w[c, s, g] * sum(CAPci.get((c, i), 0) * Ysi.get((s, i), 0) for i in activity_types) for c in classrooms)>= Msg.get((s, g), 0)* (1- z_v[s]), f"Classroom_Capacity_{s}_{g}"


    # A.15 - classroom can only be assigned to one subject
    for c in classrooms:
        model += lpSum(z[c, s] for s in subjects) <= 1, f"Single_Assignment_{c}"

    # add virtual classroom
    for s in subjects:
        for c in classrooms:
            model += z_v[s]+ z[c, s] <= 1, f"Visual_Assignment_{s}_{c}"
    
    # A.16 - only available classroom can be assigned
    for c in classrooms:
        for s in subjects:
            model += z[c, s] <= Act.get((c, week, time_slot), 0), f"Classroom_Availability_{c}_{s}"

    # A.17 - consistency constraint
    for c in classrooms:
        for s in subjects:
            if Gs.get(s, 0) == 1 :
                for g in class_courses[s]:
                    model += z[c, s] >= w[c, s, g], f"Consistency_1_{c}_{s}_{g}"
                     # 补充约束
                    model += w[c, s, g] <= 1 - z_v[s], f"Consistency_1_zv_{c}_{s}_{g}"


    # A.18 - consistency constraint
    for c in classrooms:
        for s in subjects:
            if Gs.get(s, 0) == 1 :
                model += z[c, s] <= lpSum(w[c, s, g] for g in class_courses[s]), f"Consistency_2_{c}_{s}"

    # A.19 - make sure the classroom is not shared by two groups
    for c in classrooms:
        for s in subjects:
            if Gs.get(s, 0) == 1 :
                for g1 in class_courses[s]:
                    for g2 in class_courses[s]:
                        if g1 != g2:
                            model += w[c, s, g1] + w[c, s, g2] <= 1, f"No_Shared_Classroom_{c}_{s}_{g1}_{g2}"

    # # A.20 - requirement of computer
    for c in classrooms:
        for s in subjects:
            if Cs.get(s, 0) == 1 and c not in Pc :  # 课程需要计算机但教室没有
                model += z[c, s] == 0, f"Computer_Requirement_{c}_{s}"

    # # A.21 - requirement of table
    for c in classrooms:
        for s in subjects:
            if Ts.get(s, 0) == 1 and Tc.get(c, 0) == 0:  # 课程需要桌子但教室没有
                model += z[c, s] == 0, f"Table_Requirement_{c}_{s}"
 
    # A.22 - isolation requirement
    for c in classrooms:
        for s in subjects:
            if Ic.get(c, 0) == 1:
                model += lpSum(z[c_prime, s] for c_prime in classrooms if c_prime != c) <= (1 - z[c, s]) * len(classrooms), f"Isolation_Requirement_{c}_{s}"






    model.solve()

    
    print("Week:", week)
    print("Time Slot:", time_slot)
    print("Optimization Status:", LpStatus[model.status])
    value = model.objective.value()
    print("Objective Function Value:", value)
    if model.status == 1:  # Only if an optimal solution is found
        print("\nClassroom Assignment Details:")
        assigned_classrooms = []
        if z_v[s].value() == 1:
            assigned_classrooms.append((0, s))
            print(f" - Virtual Classroom assigned to Subject {s}")
        for (c, s), var in z.items():
            if var.value() == 1:
                assigned_classrooms.append((c, s))
                print(f" - Classroom {c} assigned to Subject {s}")
    else:
        print("\n!! No feasible solution found, please check the constraints !!")
        # Output conflicting constraints (if any)
        for name, constraint in model.constraints.items():
            if not constraint.valid():
                print(f"Conflicting Constraint: {name}")

    # Collect and write results within the loop
    if model.status == 1:
        # Collect results for the current iteration
        value = model.objective.value()
        assignment_pairs = []
        for (c, s), var in z.items():
            if var.value() == 1:
                # Sort by subject ascending, classroom ascending
                assignment_pairs.append((s, c))
        for s in subjects:
            if z_v[s].value() == 1:
                assignment_pairs.append((s, 0))  # Virtual classroom

        # Generate the result string as required
        sorted_assignments = sorted(assignment_pairs, key=lambda x: (int(x[0]), int(x[1])))
        result_entries = [f"{{{s}:{c}}}" for s, c in sorted_assignments]
        result_str = ",".join(result_entries)

        # Generate subject_candidates (original logic retained)
        subject_set = {s for s, _ in sorted_assignments}
        sorted_subjects = sorted(subject_set, key=int)
        subject_candidates = "{%s}" % ",".join(map(str, sorted_subjects))
    else:
        subject_candidates = "{}"
        result_str = "{}"

    # Write to CSV (retain original writing logic)
    with open("results-virtual-classroom.csv", "a", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow([
            df.iloc[count]["Week"],
            df.iloc[count]["Time_slot"],
            df.iloc[count]["subject_candidates"],
            result_str,
            value
        ])

    # Calculate components
    real_usage = sum(z[c, s].varValue for c in classrooms for s in subjects)
    virtual_usage = sum(z_v[s].varValue for s in subjects)
    print(f"Real Classroom Usage: {real_usage}, Virtual Classroom Usage: {virtual_usage}, Total Cost: {real_usage + beta * virtual_usage}")

    results.append({
        "Week": week,
        "Time_slot": time_slot,
        "subject_candidates": subject_candidates,
        "result": result_str,
        "value": value,
        "real_usage": real_usage,
        "virtual_usage": virtual_usage,
        "total_cost": real_usage + beta * virtual_usage
    })



Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/arm64/cbc /var/folders/xg/6886p3rj5kb44k2smttjctf80000gn/T/491b3834263b40069d9419fe3ed44518-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/xg/6886p3rj5kb44k2smttjctf80000gn/T/491b3834263b40069d9419fe3ed44518-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 2273 COLUMNS
At line 8874 RHS
At line 11143 BOUNDS
At line 11717 ENDATA
Problem MODEL has 2268 rows, 573 columns and 5280 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 1.02679 - 0.00 seconds
Cgl0002I 138 variables fixed
Cgl0003I 0 fixed, 0 tightened bounds, 180 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 164 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 86 strengthened rows, 0 substitu

In [110]:
results_df = pd.DataFrame(results)
results_df

,Week,Time_slot,subject_candidates,result,value,real_usage,virtual_usage,total_cost
0,1,1,"{5,15,18}","{5:32},{15:31},{18:1}",3.0,3.0,0.0,3.0
1,1,2,{11},"{11:8},{11:49}",2.0,2.0,0.0,2.0
2,1,3,"{13,16,52}","{13:33},{16:0},{52:4},{52:6},{52:55}",8.0,4.0,1.0,8.0
3,1,4,"{13,53}","{13:23},{53:8},{53:11},{53:49},{53:55}",5.0,5.0,0.0,5.0
4,1,5,{48},{48:32},1.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...
87,11,9,{10},"{10:15},{10:30},{10:41},{10:58}",4.0,4.0,0.0,4.0
88,12,1,{2},"{2:44},{2:58}",2.0,2.0,0.0,2.0
89,12,2,{33},{33:11},1.0,1.0,0.0,1.0
90,12,6,{6},"{6:11},{6:44},{6:49}",3.0,3.0,0.0,3.0


In [112]:
sum(results_df["virtual_usage"])

9.0